In [ ]:
!pip install uv

In [ ]:
!uv pip install openai pandas arize arize-otel openinference-instrumentation-openai openai-agents

In [ ]:
!git clone https://github.com/rk-yen/customer-support-collab.git

# Secrets (Before Starting) ...
Add the following variables in the Secrets section:
* ARIZE_API_KEY
* ARIZE_SPACE_ID
* OPENAI_API_KEY

In [ ]:
from pathlib import Path
import sys

import pandas as pd

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "workshop_helpers").exists():
    candidates = [path for path in REPO_ROOT.iterdir() if path.is_dir() and (path / "workshop_helpers").exists()]
    if not candidates:
        raise FileNotFoundError("Could not find a repo folder containing workshop_helpers")
    REPO_ROOT = candidates[0]

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from workshop_helpers.backend import (
    TOOLS,
    hydrate_backend_from_dataset,
    run_support_agent_async,
    snapshot_backend,
)
from workshop_helpers.data import DATASET
from workshop_helpers.experiments import (
    build_v2_support_snapshot,
    format_checklist_rows,
    prepare_experiment_bundle,
    production_readiness_checklist,
    run_experiment,
    summarize_dataset,
)
from workshop_helpers.metrics import compare_scores
from workshop_helpers.scenarios import (
    run_context_agent,
    run_raw_llm,
)


In [ ]:
from workshop_helpers.setup import setup_clients

env = setup_clients()
client = env["client"]
ARIZE_SPACE_ID = env["arize_space_id"]
ARIZE_API_KEY = env["arize_api_key"]

---
## Step 1 — Raw LLM Call

The simplest possible baseline: a two-sentence system prompt, no customer data.
This is what most teams' v0 looks like — pasting into ChatGPT.

In [ ]:
PROMPT_V1 = "You are a customer support agent. Help the customer."
CUSTOMER_MESSAGE = (
    "I bought a laptop stand about 35 days ago and its still in the box, "
    "completely unopened. I would like to return it."
)

output_v1 = run_raw_llm(client, CUSTOMER_MESSAGE, PROMPT_V1)

print("V1 (raw LLM):")
print("-" * 50)
print(output_v1)
print("-" * 50)


**What is missing?**  The model has no idea who the customer is, what they ordered,
or what your return policy says. It can only ask for more information — or worse,
make something up.

## Update PROMPT_V1
Use https://platform.openai.com/chat/edit?models=gpt-4o-mini&optimize=true and update PROMPT_V1

### Objective
_Improve this baseline customer support system prompt for cases where the model only has the customer's message and no backend access. The response should be empathetic, concise, and honest about uncertainty. It must not invent order details, policy details, dates, or actions taken. When information is missing, it should ask only the single most useful follow-up question. Optimize for better tone and clearer issue handling, while keeping the limitations of a raw prompt-only assistant explicit._

### User prompt
```
Objective:
{objective}

Current prompt:
{baseline_prompt}

Rewrite the prompt so the model is empathetic, concise, and honest about uncertainty.
If order/account details are missing, it should ask only the single most important follow-up question.
```

---
## Step 2 — Add Context

Inject the customer's account data directly into the prompt.
No tools yet — just better information.
This is the **prompted agent** pattern.

In [ ]:
import json
def build_context_message(customer_message: str, source_data: dict) -> str:
    context_block = json.dumps(source_data, indent=2)
    return f"Customer context (internal):\n{context_block}\n\nCustomer message:\n{customer_message}"

CUSTOMER_MESSAGE = (
    "I bought a laptop stand about 35 days ago and its still in the box, "
    "completely unopened. I would like to return it."
)

SOURCE_DATA = {
    "customer_name": "Elena Vasquez",
    "customer_id": "CUST_6601",
    "order_id": "ORD_8814",
    "product_name": "ErgoRise Pro Laptop Stand",
    "order_date": "2024-02-16",
    "days_since_order": 35,
    "order_total": 69.99,
    "order_status": "delivered",
    "return_policy_days": 30,
    "account_status": "active",
    "notes": "Order is 5 days outside the standard return window. Supervisor escalation available.",
}

V2_CONTEXT = build_v2_support_snapshot(
    {
        "category": "returns",
        "source_data": SOURCE_DATA,
    }
)

PROMPT_V2 = (
    "You are a customer support agent looking at an internal support snapshot. "
    "Use the provided context carefully and be specific. "
    "Do not claim that you already issued a refund, sent a label, escalated a case, or made any backend change. "
    "If the context suggests the customer is eligible for help, explain the best next step clearly and accurately. "
    "Never cite figures or dates not in the context."
)

print("Context passed to the model:")
customer_message_with_context = build_context_message(CUSTOMER_MESSAGE, V2_CONTEXT)
print(customer_message_with_context)
print()

output_v2 = run_context_agent(client, customer_message_with_context, PROMPT_V2)

print("V2 (with context):")
print("-" * 50)
print(output_v2)
print("-" * 50)


### Scoring — three metrics we will reuse throughout

| Metric | Type | What it checks |
|--------|------|----------------|
| `source_grounding` | Programmatic | No invented dollar amounts |
| `tone_quality` | AI judge | Empathetic and professional |
| `issue_resolved` | AI judge | Actually addresses the problem |

In [ ]:
scorecard = pd.DataFrame(
    compare_scores(
        client,
        {
            "V1 raw LLM": output_v1,
            "V2 + context": output_v2,
        },
        source_data=SOURCE_DATA,
    )
)

display(scorecard[["variant", "grounding_ok", "tone", "resolved", "total"]])


---
## Step 3 — Tool-Calling Agent (OpenAI Agents SDK)

V2 trusts whatever we inject. If `days_since_order` is stale in the database,
the model uses the wrong value. Policy arithmetic ("is this within 30 days?") is
just math — a model should not guess at it.

**Tools** let the model call real functions: it decides *what* to look up,
and *when* to act. That is the difference between a prompted assistant and an agent.

We use the [OpenAI Agents SDK](https://openai.github.io/openai-agents-python/),
which handles the tool-calling loop, tracing, and function schema generation
from Python type hints and docstrings.

```
  Customer message
       |
       v
  [ gpt-4o-mini ]  -- get_customer_profile(CUST_6601) -->  ORDER_DB + ACCOUNT_DB
       |                                                          |
       |          <-- profile: orders, subscription, app --------+
       |
       +-- check_return_eligibility(ORD_8814) --> 35 days, outside window by 5
       |
       v
  Declines + offers supervisor escalation
```

In [ ]:
backend_snapshot = snapshot_backend()

print("Simulated systems ready:")
for label, value in backend_snapshot.items():
    print(f"  {label}: {value}")


In [ ]:
print("Tools available to the agent:")
for tool in TOOLS:
    print(f"  - {tool.name}")


In [ ]:
AUTHENTICATED_CUSTOMER_ID = SOURCE_DATA["customer_id"]

agent_result = await run_support_agent_async(
    customer_message=CUSTOMER_MESSAGE,
    authenticated_customer_id=AUTHENTICATED_CUSTOMER_ID,
)
output_v3 = agent_result.final_output
tool_calls = [
    raw.name
    for item in agent_result.new_items
    for raw in [getattr(item, "raw_item", None)]
    if raw and hasattr(raw, "name")
]

print("V3 (Agents SDK):")
print("-" * 50)
print(output_v3)
print("-" * 50)
print()
print("Tool call chain:")
for name in tool_calls:
    print(f"  -> {name}()")


In [ ]:
scorecard = pd.DataFrame(
    compare_scores(
        client,
        {
            "V1 raw LLM": output_v1,
            "V2 context": output_v2,
            "V3 tools": output_v3,
        },
        source_data=SOURCE_DATA,
    )
)

display(scorecard[["variant", "grounding_ok", "tone", "resolved", "total"]])

print("What tools add over context injection:")
print("  - Policy arithmetic is code, not inference")
print("  - Actions become explicit system calls, not promises in prose")
print("  - The same agent shell can grow across returns, billing, access, and subscriptions")


---
## Step 4 — Arize Datasets + Experiments

One demo case builds intuition. **Experiments** answer the harder question:
does the improvement hold across all 20 inputs, including edge cases?

We use the **Arize Datasets client** to:
1. Upload the 20 cases as a named, versioned **dataset**
2. Run each of the three approaches as a named **experiment**
3. Compare all three in the Arize UI side-by-side

Every task call and evaluator call is also traced — visible in Arize AX
under the project `Workiva-ProblemFirst-Workshop`.

In [ ]:
dataset_summary = summarize_dataset(DATASET)

print(f"Dataset: {dataset_summary['scenario_count']} scenarios")
print(f"  Standard:   {dataset_summary['standard_count']}")
print(f"  Edge cases: {dataset_summary['edge_case_count']}")
print(f"  Categories: {', '.join(dataset_summary['categories'])}")

pd.DataFrame(DATASET)[["scenario_id", "category", "difficulty", "is_edge_case"]].head()


In [ ]:
backend_snapshot = hydrate_backend_from_dataset(DATASET)
print("Backend refreshed from the 20-case dataset:")
for label, value in backend_snapshot.items():
    print(f"  {label}: {value}")


In [ ]:
experiment_bundle = prepare_experiment_bundle(
    client=client,
    arize_api_key=ARIZE_API_KEY,
    arize_space_id=ARIZE_SPACE_ID,
    dataset=DATASET,
    prompt_v1=PROMPT_V1,
    prompt_v2=PROMPT_V2,
)

arize_client = experiment_bundle["client"]
DATASET_ID = experiment_bundle["dataset_id"]

print(f"Dataset name: {experiment_bundle['dataset_name']}")
print(f"Dataset ID:   {DATASET_ID}")
print(f"Rows:         {experiment_bundle['row_count']}")
print(f"Created now:  {experiment_bundle['created']}")


In [ ]:
tasks = experiment_bundle["tasks"]

print("Tasks defined:")
for name in tasks:
    print(f"  - {name}")


In [ ]:
EVALUATORS = experiment_bundle["evaluators"]

print("Evaluators ready:")
for evaluator in EVALUATORS:
    print(f"  - {evaluator.__class__.__name__}")


In [ ]:
print("Running experiment 1/3: v1-raw-llm...")
run_v1 = run_experiment(
    arize_client=arize_client,
    dataset_id=DATASET_ID,
    name_prefix="v1-raw-llm",
    task=tasks["task_v1"],
    evaluators=EVALUATORS,
)
exp_v1 = run_v1["experiment"]
df_v1 = run_v1["results_df"]
print(f"  Done. Experiment ID: {exp_v1.id}")
print(f"  Rows evaluated: {len(df_v1)}")


In [ ]:
print("Running experiment 2/3: v2-context-agent...")
run_v2 = run_experiment(
    arize_client=arize_client,
    dataset_id=DATASET_ID,
    name_prefix="v2-context-agent",
    task=tasks["task_v2"],
    evaluators=EVALUATORS,
)
exp_v2 = run_v2["experiment"]
df_v2 = run_v2["results_df"]
print(f"  Done. Experiment ID: {exp_v2.id}")
print(f"  Rows evaluated: {len(df_v2)}")


In [ ]:
print("Running experiment 3/3: v3-tool-agent (async tasks, takes a few minutes)...")
run_v3 = run_experiment(
    arize_client=arize_client,
    dataset_id=DATASET_ID,
    name_prefix="v3-tool-agent",
    task=tasks["task_v3"],
    evaluators=EVALUATORS,
)
exp_v3 = run_v3["experiment"]
df_v3 = run_v3["results_df"]
print(f"  Done. Experiment ID: {exp_v3.id}")
print(f"  Rows evaluated: {len(df_v3)}")


---
## Step 5 — Production Readiness

Use the experiment results in Arize to fill in this checklist.
The goal is not 10/10 — it is knowing which gaps remain before you decide
on the right level of autonomy for your v1 launch.

In [ ]:
for row in format_checklist_rows(production_readiness_checklist()):
    print(row)
